# URL Word2Vec embedding (demo)

This notebook tests `hf_phishing_url.UrlWord2VecVectorizer`, which trains a Word2Vec model over URL tokens and pools token vectors into a fixed-size embedding per URL.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Allow running the notebook without installing the package.
sys.path.append(str(Path.cwd().parent / "src"))

## Fit + transform a small URL list


In [ ]:
from hf_phishing_url import UrlWord2VecVectorizer

urls = [
    "https://accounts.google.com/signin/v2/identifier",
    "https://mail.google.com/mail/u/0/",
    "paypal-secure.example.net/verify",
    "http://example.com/login.php?session=123",
    "https://microsoftonline.com/common/oauth2/v2.0/authorize",
]

vec = UrlWord2VecVectorizer(
    sources=("host", "path"),
    vector_size=32,
    window=5,
    min_count=1,
    epochs=25,
    pooling="mean",
    normalize=True,
)

X = vec.fit_transform(urls)

display(X.shape)

In [10]:
import numpy as np

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    denom = float(np.linalg.norm(a) * np.linalg.norm(b))
    return float(a @ b / denom) if denom else 0.0

if "X" in globals():
    # Similar URLs should usually be closer than unrelated ones.
    print("google signin vs google mail:", cosine(X[0], X[1]))
    print("google signin vs paypal verify:", cosine(X[0], X[2]))


google signin vs google mail: 0.3157085180282593
google signin vs paypal verify: -0.395611435174942


## Fit on a sample from the Hugging Face dataset

In [12]:
try:
    from datasets import load_dataset

    ds = load_dataset("pirocheto/phishing-url")
    train_urls = ds["train"]["url"][:5000]

    vec2 = UrlWord2VecVectorizer(
        sources=("host", "path"),
        vector_size=64,
        min_count=2,
        epochs=10,
        pooling="mean",
        normalize=True,
    )
    X_train = vec2.fit_transform(train_urls)
    print("Train matrix:", X_train.shape)
except Exception as e:
    print("Could not load/fit (maybe no network / not cached / missing gensim):")
    print(type(e).__name__, str(e)[:250])

Train matrix: (5000, 64)


In [15]:
v = vec2.transform(train_urls)

display(v.shape)
display(v)

(5000, 64)

array([[ 0.10150607,  0.00505757,  0.15335003, ...,  0.0157429 ,
        -0.07208212,  0.18225746],
       [ 0.15811133, -0.03152541,  0.18575446, ...,  0.06030754,
        -0.02954293,  0.1800625 ],
       [ 0.04671426, -0.09668355,  0.13784888, ..., -0.03481631,
        -0.00708509,  0.14443593],
       ...,
       [ 0.00348876, -0.11307894,  0.10603636, ..., -0.08196753,
         0.02779901,  0.11893257],
       [-0.02090456, -0.09503075,  0.00096985, ..., -0.12426559,
         0.02484026,  0.08913652],
       [ 0.08267056,  0.01178314,  0.24517243, ..., -0.0386251 ,
        -0.02838923,  0.18856564]], shape=(5000, 64), dtype=float32)